# v10 · KCF-1: HyperWebster address → HKDF key material
`PtolemyDesktop/Kryptos/kcf.py` (git 2026-05-03).

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
import sys
sys.path.insert(0, os.path.join(PROJ, "PtolemyDesktop"))
kcf = load("PtolemyDesktop/Kryptos/kcf.py", "kcf")
K = kcf.KCF(face_id="callimachus")
k1 = K.derive("hiraeth"); k2 = K.derive("hiraeth"); k3 = K.derive("hiraeth ")
print(k1.hex()[:32], len(k1), "| deterministic:", k1 == k2, "| trailing space changes key:", k1 != k3)

b2adba56a5b6d342785e1ea0657d5486 32 | deterministic: True | trailing space changes key: True


## `hw_address` is not the bijection
It is a plain base-N numeral with a zero digit (index 0 = first charset symbol), and unknown characters fall back to `ord(ch) % N`. Both make distinct labels share an address.

In [3]:
cs0 = K.charset[0]
print("first charset symbol:", repr(cs0))
print("hw_address(cs0+'x') == hw_address('x'):", K.hw_address(cs0 + "x") == K.hw_address("x"))
print("hw_address('é') == hw_address('%s'):" % chr(ord("é") % 97 and 0) if False else "", end="")
alias = next(c for c in K.charset if K.charset.index(c) == ord("é") % len(K.charset))
print("hw_address('é') == hw_address(%r):" % alias, K.hw_address("é") == K.hw_address(alias))
assert K.hw_address(cs0 + "x") == K.hw_address("x")

first charset symbol: ' '
hw_address(cs0+'x') == hw_address('x'): True
hw_address('é') == hw_address('I'): True


## The 'permutation offset' is a hash of the charset string, folded modulo a Mersenne prime
`_permute` XORs the address with 64 bits of SHA-256(charset), multiplies by 2⁶¹−1 and reduces mod 2⁸⁹−1. The output is below 2⁸⁹ whatever the label length; HKDF then turns it into key bytes.

In [4]:
big = "x" * 200
a = K.hw_address(big); m = K._permute(a)
print("address bits:", a.bit_length(), "| _permute output bits:", m.bit_length(), "(< 89)")
assert m.bit_length() <= 89
# two labels far apart in address space can still differ; the map is not the N! permutation index charset.py describes
print("charset-order dependence: same label, charset reversed ->",
      kcf.KCF("callimachus", charset=K.charset[::-1]).derive("hiraeth") != k1)

address bits: 1317 | _permute output bits: 88 (< 89)
charset-order dependence: same label, charset reversed -> True
